# Libraries

In [70]:
import pandas as pd
import seaborn as sns
from functools import reduce

# Resampling Libs
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

# Functions

## Resample

In [71]:
def fun_res(dfi):

    X = dfi.drop("Study_Status_Bin", axis = 1)
    y = dfi["Study_Status_Bin"]

    X_train_tts, X_test_tts, y_train_tts, y_test_tts = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

    res = RandomUnderSampler(sampling_strategy = 'auto', random_state = 42)
    X_train_res, y_train_res = res.fit_resample(X_train_tts, y_train_tts) 
    
    X_train_res = pd.DataFrame(X_train_res)

    return X_train_res, y_train_res


## Fun_Sparse

In [72]:
def fun_sparse(i, dfi, categ_cols):
    pivot_tables = []

    for col in categ_cols:
        if col not in dfi.columns:
            # αν δεν υπάρχει, φτιάξε dummy στήλη γεμάτη NaN/0
            dfi[col] = pd.Series([0]*len(dfi), index=dfi.index)

        pivot_table = pd.pivot_table(
            data = dfi,
            index = col,
            columns = "Study_Status_Bin",
            aggfunc = "size",
            fill_value = 0,
            observed = False
        ).reset_index()

        # Change Column/Element Names of Pivot
        pivot_table['Variables'] = col + ' = ' + pivot_table[col].astype(str)
        pivot_table.drop(columns=[col], inplace=True)

        # Reindex Column of value
        final_cols = ['Variables'] + [c for c in pivot_table.columns if c != 'Variables']
        pivot_table = pivot_table[final_cols]

        pivot_tables.append(pivot_table)

    # Merge all pivots
    pivot_merged = pd.concat(pivot_tables, ignore_index=True)
    pivot_merged = pivot_merged.rename(columns={0: f'0 - df{i}', 1: f'1 - df{i}'})
    # pivot_merged = pivot_merged.drop(columns = [f'0 - df{i}'], axis = 1) --> EPV regarding the minority (1) class only
    # However, the variables are the same even if include majority (0) in pivots --> keep majority for exploratory analysis
    
    return pivot_merged


## Fun_Zeros

In [73]:
def fun_zeros(pivot_merged, count, missing):

    num_cols = pivot_merged.select_dtypes(include='number')
    mask = num_cols < count
    if missing == True:
        mask |= num_cols.isna()
    sparse = pivot_merged[mask.any(axis=1)]
    return sparse

# Load Unmerged Data
- Data loaded haven't dropped first after dummies. All levels need checking
- Only on train data. 
- Test data are 'unseen' --> no sparsity check. 

In [74]:
df1 = pd.read_pickle(r".\df_dummies_unmerged\df1_dummies_unmerged.pkl")
df2 = pd.read_pickle(r".\df_dummies_unmerged\df2_dummies_unmerged.pkl")
df3 = pd.read_pickle(r".\df_dummies_unmerged\df3_dummies_unmerged.pkl")
df4 = pd.read_pickle(r".\df_dummies_unmerged\df4_dummies_unmerged.pkl")

## Resample

In [75]:
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)
X_train4.head()

,Enrollment_Counts,Funder_Counts,Funder_Counts_Log,Intervention_Type_Counts,Intervention_Route_Counts,Placebo_Bin,Standard_Care_Bin,Healthy_Bin,Covid_19_Bin,Adverse_Counts,...,Primary_Purpose_List_SCREENING,Primary_Purpose_List_SUPPORTIVE_CARE,Primary_Purpose_List_TREATMENT,Continents_List_Africa,Continents_List_Asia,Continents_List_Cont_Other,Continents_List_Europe,Continents_List_North America,Continents_List_Oceania,Continents_List_South America
1933,454,1,0.693147,1,1,0,0,0,0,27,...,0,0,1,0,0,0,1,0,0,0
7496,64,3,1.386294,1,1,1,0,1,0,0,...,0,1,0,0,0,1,0,0,0,0
3966,619,1,0.693147,1,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2389,211,2,1.098612,1,1,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
8416,612,2,1.098612,1,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


## Create X_y dfs

In [76]:
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

display(X_train1.shape)
display(X_train2.shape)
display(X_train3.shape)
display(X_train4.shape)

(5092, 169)

(8678, 170)

(3854, 168)

(3694, 169)

## Unique cols

In [77]:
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

# Columns missing from the dfs
for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    print(f" X_train{i}: shape={df.shape}")
    print(f" Missing cols: {missing if missing else 'None'}")
    print(f" Extra cols:   {extra if extra else 'None'}\n")


172
 X_train1: shape=(5092, 169)
 Missing cols: {'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Fluids and Secretions', 'Conditions_Detail_List_Chemical Actions and Uses'}
 Extra cols:   None

 X_train2: shape=(8678, 170)
 Missing cols: {'Conditions_Detail_List_Fluids and Secretions', 'Conditions_Detail_List_Chemical Actions and Uses'}
 Extra cols:   None

 X_train3: shape=(3854, 168)
 Missing cols: {'Conditions_Detail_List_Chemical Actions and Uses', 'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Human Activities', 'Conditions_Detail_List_Information Science'}
 Extra cols:   None

 X_train4: shape=(3694, 169)
 Missing cols: {'Conditions_Detail_List_Hemic and Immune Systems', 'Conditions_Detail_List_Human Activities', 'Conditions_Detail_List_Fluids and Secretions'}
 Extra cols:   None



## Pivots

### List/Categ/Bin Pivot

In [78]:
# Categ Pivot
categ_cols = [col for col in all_unique_cols if '_Categ' in col or '_Bin' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, categ_cols)
pivot_merged2 = fun_sparse(2, df2_train, categ_cols)
pivot_merged3 = fun_sparse(3, df3_train, categ_cols)
pivot_merged4 = fun_sparse(4, df4_train, categ_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_categ = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_categ

# List Pivot
list_cols = [col for col in all_unique_cols if '_List' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, list_cols)
pivot_merged2 = fun_sparse(2, df2_train, list_cols)
pivot_merged3 = fun_sparse(3, df3_train, list_cols)
pivot_merged4 = fun_sparse(4, df4_train, list_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_list = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_list


#### Zeros Categ/Bin


In [79]:
sparse_categ_train = fun_zeros(pivot_merged_train_categ, 20, False) # No categorical cols were used. 
display(sparse_categ_train)

sparse_list_train = fun_zeros(pivot_merged_train_list, 20, True)
display(sparse_list_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
23,Adverse_System_List_Adv_Syst_Immune System = 1,59.0,87.0,428.0,298.0,277.0,163.0,63.0,19.0
65,Conditions_Detail_List_Biological Phenomena = 1,2.0,1.0,3.0,5.0,0.0,2.0,3.0,4.0
69,Conditions_Detail_List_Cell Physiological Phen...,3.0,4.0,13.0,17.0,8.0,13.0,19.0,9.0
71,Conditions_Detail_List_Cells = 1,1.0,1.0,2.0,4.0,NaN,NaN,NaN,NaN
73,Conditions_Detail_List_Chemical Actions and Us...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0
77,Conditions_Detail_List_Circulatory and Respira...,NaN,NaN,3.0,4.0,2.0,0.0,0.0,6.0
83,Conditions_Detail_List_Diagnosis = 1,30.0,13.0,61.0,39.0,25.0,15.0,39.0,36.0
87,Conditions_Detail_List_Education = 1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN
91,Conditions_Detail_List_Environment and Public ...,1.0,2.0,10.0,11.0,4.0,4.0,10.0,14.0
98,Conditions_Detail_List_Genetic Phenomena = 1,3.0,0.0,2.0,2.0,NaN,NaN,NaN,NaN


# Check on Merged

## Load Merged Data

In [80]:
df1 = pd.read_pickle(r".\df_dummies\df1_dummies.pkl")
df2 = pd.read_pickle(r".\df_dummies\df2_dummies.pkl")
df3 = pd.read_pickle(r".\df_dummies\df3_dummies.pkl")
df4 = pd.read_pickle(r".\df_dummies\df4_dummies.pkl")

## Functions

In [81]:
# Resample
X_train1, y_train1 = fun_res(df1)
X_train2, y_train2 = fun_res(df2)
X_train3, y_train3 = fun_res(df3)
X_train4, y_train4 = fun_res(df4)

# Create X_train y_train
df1_train = pd.concat([X_train1, y_train1], axis=1)  # X_train y_train have same index
df2_train = pd.concat([X_train2, y_train2], axis=1)  # X_train y_train have same index
df3_train = pd.concat([X_train3, y_train3], axis=1)  # X_train y_train have same index
df4_train = pd.concat([X_train4, y_train4], axis=1)  # X_train y_train have same index

# Unique Columns
dfs = [X_train1, X_train2, X_train3, X_train4]
all_unique_cols = set().union(*(df.columns for df in dfs))
print(f"{len(all_unique_cols)}")

for i, df in enumerate(dfs, start=1):
    missing = all_unique_cols - set(df.columns)
    extra = set(df.columns) - all_unique_cols
    # print(f" X_train{i}: shape={df.shape}")
    # print(f" Missing cols: {missing if missing else 'None'}")
    # print(f" Extra cols:   {extra if extra else 'None'}\n")

72


## Interaction

In [86]:
# Columns
inter_cols = [col for col in all_unique_cols if '_x_' in col]  

pivot_merged1 = fun_sparse(1, df1_train, inter_cols)
pivot_merged2 = fun_sparse(2, df2_train, inter_cols)
pivot_merged3 = fun_sparse(3, df3_train, inter_cols)
pivot_merged4 = fun_sparse(4, df4_train, inter_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_inter = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 
# pivot_merged_train_inter


### Zeros Inter


In [87]:
sparse_inter_train = fun_zeros(pivot_merged_train_inter, 20, True)
display(sparse_inter_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
11,Enrollment_x__Endocrine System = 1,91,51,139,47,84,41,76,16
13,Enrollment_x__Eye = 1,14,5,79,25,38,15,25,6
17,"Enrollment_x__Hereditary, Neonatal, Abnormalit...",42,14,117,54,32,23,15,10
19,"Enrollment_x__Musculoskeletal, Neural = 1",65,15,226,58,81,31,159,39
21,Enrollment_x__Neoplasms = 1,184,216,598,213,154,81,32,18
27,"Enrollment_x__Phenomena, Processes = 1",44,13,109,20,30,10,54,13
35,"Enrollment_x__Stomatognathic, Otorhinolaryngol...",13,3,68,10,29,5,38,4


## Categ/Bin/List Pivot

In [88]:
# Pivot_Categ/Bin
categ_cols = [col for col in all_unique_cols if '_Categ' in col or '_Bin' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, categ_cols)
pivot_merged2 = fun_sparse(2, df2_train, categ_cols)
pivot_merged3 = fun_sparse(3, df3_train, categ_cols)
pivot_merged4 = fun_sparse(4, df4_train, categ_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_categ = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 

# Pivot_List
list_cols = [col for col in all_unique_cols if '_List' in col and 'Study_Status_Bin' not in col]  

pivot_merged1 = fun_sparse(1, df1_train, list_cols)
pivot_merged2 = fun_sparse(2, df2_train, list_cols)
pivot_merged3 = fun_sparse(3, df3_train, list_cols)
pivot_merged4 = fun_sparse(4, df4_train, list_cols)

all_pivots = [pivot_merged1, pivot_merged2, pivot_merged3, pivot_merged4]
pivot_merged_train_list = reduce(lambda left, right: pd.merge(left, right, on="Variables", how="outer"), all_pivots) 


### Zeros Categ/Bin/List


In [89]:
sparse_categ_train = fun_zeros(pivot_merged_train_categ, 20, False) # No categorical cols were used. 
display(sparse_categ_train) # ok

sparse_list_train = fun_zeros(pivot_merged_train_list, 20, True)
display(sparse_list_train)

Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4


Study_Status_Bin,Variables,0 - df1,1 - df1,0 - df2,1 - df2,0 - df3,1 - df3,0 - df4,1 - df4
